# Live Scrape → RAG (Ollama embeddings + Claude answers)

Feed scraping targets in — directly, from agent tools, or from a **Kafka topic** —
scrape the pages, embed them with **Ollama**, index them, and ask questions answered
by **Claude** with retrieval-augmented generation.

Built on `org.agentic.flink.rag.KnowledgeBase` and the `scrape_url` / `ask_knowledge_base`
agent tools.

## Prereqs
- Framework jar built: `mvn -DskipTests package`
- **Ollama** running with an embedding model: `podman exec ollama ollama pull nomic-embed-text`
- **Kafka** running on `localhost:9092` (section 5 only) — `podman compose -f docker-compose-kafka.yml up -d`
- For section 3 (answers): `export ANTHROPIC_API_KEY=sk-ant-...` before launching Jupyter
- Python deps: `pip install -e python/ kafka-python`

## 1. Bootstrap the JVM + build the knowledge base

In [1]:
import os, pathlib, subprocess

repo_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()

if 'AGENTIC_FLINK_JAR' not in os.environ:
    jars = [j for j in sorted((repo_root / 'target').glob('agentic-flink-*.jar'))
            if 'original' not in j.name and 'sources' not in j.name]
    assert jars, 'build the jar first: mvn -DskipTests package'
    os.environ['AGENTIC_FLINK_JAR'] = str(jars[-1])

cp_cache = repo_root / 'target' / 'runtime-classpath.txt'
if not cp_cache.exists():
    print('Resolving runtime classpath via Maven (one-off)...')
    subprocess.run(['mvn', '-q', 'dependency:build-classpath',
                    f'-Dmdep.outputFile={cp_cache}', '-Dmdep.includeScope=runtime'],
                   cwd=repo_root, check=True)
extra_jars = [j for j in cp_cache.read_text().strip().split(':') if j]

import agentic_flink as af
af.start_jvm(extra_jars=extra_jars)
from agentic_flink import jclass
print('JVM started:', af.is_started())

JVM started: True


In [2]:
OLLAMA_URL = os.environ.get('OLLAMA_URL', 'http://localhost:11434')
EMBED_MODEL = os.environ.get('EMBED_MODEL', 'nomic-embed-text')
EMBED_DIM = int(os.environ.get('EMBED_DIM', '768'))
ANTHROPIC_KEY = os.environ.get('ANTHROPIC_API_KEY')

KB = jclass('org.agentic.flink.rag.KnowledgeBase')
builder = KB.builder().withOllamaEmbedding(OLLAMA_URL, EMBED_MODEL, EMBED_DIM)
if ANTHROPIC_KEY:
    builder = builder.withClaude(ANTHROPIC_KEY)
    print('Chat: Claude enabled')
else:
    print('Chat: no ANTHROPIC_API_KEY set — section 3 (ask) will be skipped')
kb = builder.build()
print('Embeddings: Ollama', EMBED_MODEL, f'({EMBED_DIM}-dim) at', OLLAMA_URL)

Chat: no ANTHROPIC_API_KEY set — section 3 (ask) will be skipped
Embeddings: Ollama nomic-embed-text (768-dim) at http://localhost:11434


## 2. Feed scraping targets directly

Any reachable URL works. Set `SCRAPE_URLS` (comma-separated) to override the defaults —
e.g. point at a docs site you want to ask questions about.

In [3]:
default_urls = [
    'https://flink.apache.org/what-is-flink/flink-architecture/',
    'https://en.wikipedia.org/wiki/Apache_Flink',
]
urls = [u for u in os.environ.get('SCRAPE_URLS', ','.join(default_urls)).split(',') if u]

for u in urls:
    r = kb.ingestUrl(u)
    status = f"{r.chunks} chunks" if r.ok else f"FAILED: {r.error}"
    print(f'  {u}  ->  {status}  ({r.title})')
print('\nindexed vectors:', kb.stats().get('total_vectors'))

  http://localhost:8077/flink.html  ->  1 chunks  (Apache Flink Overview)
  http://localhost:8077/agents.html  ->  1 chunks  (Agentic Flink)

indexed vectors: 2


## 3. Semantic search over the scraped content

In [4]:
query = 'How does Flink recover state after a failure?'
for p in kb.search(query, 4):
    label = p.title or p.id
    print(f'  [{p.score:.3f}] {label}')
    print(f'        {str(p.text)[:140].strip()}...')

  [0.599] Apache Flink Overview
        Apache Flink Apache Flink is an open-source, distributed stream-processing framework for stateful computations over unbounded and bounded da...
  [0.589] Agentic Flink
        Agentic Flink Agentic Flink layers LLM-powered agents on top of Apache Flink. Memory lives in Flink keyed state by default, and chat, embedd...


## 4. Ask a question (RAG answer from Claude)

Retrieves the top passages and asks Claude to answer grounded in them, with citations.
Skipped automatically if no `ANTHROPIC_API_KEY` was set.

In [5]:
if not ANTHROPIC_KEY:
    print('⚠ No ANTHROPIC_API_KEY — skipping. Set it and re-run cell 1+2 to enable.')
else:
    ans = kb.ask('What guarantees does Flink give about state consistency, and how?', 4)
    print('ANSWER:\n', ans.text)
    print('\nSOURCES:')
    for p in ans.sources:
        print(f'  - {p.title or p.id}  ({p.url})')

⚠ No ANTHROPIC_API_KEY — skipping. Set it and re-run cell 1+2 to enable.


## 5. Agent-callable scraping tools

The same knowledge base exposed as `ToolExecutor`s an agent can call: `scrape_url`
(fetch + index) and `ask_knowledge_base` (retrieve + answer). This is how one agent
feeds targets and another asks questions.

In [6]:
from java.util import HashMap

ScrapeUrlTool = jclass('org.agentic.flink.tools.rag.ScrapeUrlTool')
scrape_tool = ScrapeUrlTool(kb)

args = HashMap(); args.put('url', urls[0])
result = scrape_tool.execute(args).get()
print('scrape_url tool ->', dict(result))

# ask_knowledge_base tool (only meaningful with Claude configured)
if ANTHROPIC_KEY:
    KnowledgeQueryTool = jclass('org.agentic.flink.tools.rag.KnowledgeQueryTool')
    ask_tool = KnowledgeQueryTool(kb)
    q = HashMap(); q.put('question', 'Summarize Flink in one sentence.'); q.put('top_k', 3)
    print('\nask_knowledge_base tool ->', str(ask_tool.execute(q).get().get('answer')))

scrape_url tool -> {'chunks_indexed': 1, 'title': 'Apache Flink Overview', 'ok': True, 'url': 'http://localhost:8077/flink.html'}


## 6. Feed scraping targets from a Kafka topic

Produce URLs onto a topic and consume them into the knowledge base — the pattern for
one service (or agent) streaming scrape targets to another. Requires Kafka on
`localhost:9092`.

In [7]:
import json
try:
    from kafka import KafkaProducer, KafkaConsumer
    from kafka.errors import NoBrokersAvailable
    TOPIC = 'scrape-targets'
    targets = os.environ.get('KAFKA_SCRAPE_URLS', ','.join(urls)).split(',')

    producer = KafkaProducer(bootstrap_servers='localhost:9092',
                             value_serializer=lambda v: json.dumps(v).encode())
    for u in targets:
        producer.send(TOPIC, {'url': u})
    producer.flush()
    print(f'produced {len(targets)} targets to topic {TOPIC!r}')

    consumer = KafkaConsumer(TOPIC, bootstrap_servers='localhost:9092',
                             auto_offset_reset='earliest', group_id='kb-ingest',
                             consumer_timeout_ms=8000,
                             value_deserializer=lambda b: json.loads(b.decode()))
    seen = set()
    for msg in consumer:
        u = msg.value['url']
        if u in seen:
            continue
        seen.add(u)
        r = kb.ingestUrl(u)
        print(f'  consumed {u} -> {r.chunks} chunks (ok={r.ok})')
    consumer.close(); producer.close()
    print('\nindexed vectors after Kafka feed:', kb.stats().get('total_vectors'))
except NoBrokersAvailable:
    print('⚠ Kafka not reachable on localhost:9092 — skipping. Start it with:')
    print('  podman compose -f docker-compose-kafka.yml up -d')

produced 2 targets to topic 'scrape-targets'


  consumed http://localhost:8077/flink.html -> 1 chunks (ok=True)
  consumed http://localhost:8077/agents.html -> 1 chunks (ok=True)



indexed vectors after Kafka feed: 2


## Done

You scraped pages (directly, via the `scrape_url` agent tool, and from a Kafka topic),
embedded them with Ollama, indexed them in-memory, and — with a Claude key — answered
questions grounded in the scraped content with citations.

Swap the in-memory store for `qdrant`/`milvus`/`pgvector` by building the KnowledgeBase
with `.withVectorStore(StorageFactory.createVectorStore("qdrant", config))`.